# 11 · Mock coding test A — regression, then multi-class classification

**Rules — treat this as the real thing.**

- **150 minutes**, timer running, one sitting.
- A fresh Jupyter session, **no autocomplete, no internet, no AI, no other notebooks from this folder**. What you *may* use (as in the real test): your own homework solution, the official course materials, and library documentation.
- The offline documentation is always one cell away: `help(nn.CrossEntropyLoss)`, `nn.Linear?`, `np.nanmedian?`. Practise using it now — it is the only "tool" you will have.
- Write your solutions between the `# implement your solution BELOW` / `ABOVE` lines. Each sub-task ends with a `tester.test(...)` call that prints `OK` or `FAILED` with a reason. As in the homework, `OK` means *possibly* correct.
- Vectorised solutions are expected; Python loops only for parsing text, for epochs/batches, and for a handful of printed examples or plot annotations.

When the time is up: stop, note which sub-tasks were unfinished and **what exactly held you up**, then compare with the solution notebook.

---
**The data.** `bikes.csv` — hourly records of a bike-rental station. Fields are separated by **semicolons**; the first line holds the names:

`hour;temp_c;humidity;wind_kmh;weekday;season;holiday;rentals`

- `hour` 0–23, `weekday` 0 (Monday) – 6 (Sunday), `holiday` 0/1
- `season` is text: `winter`, `spring`, `summer`, `autumn`
- `temp_c`, `humidity`, `wind_kmh` are sensor readings and **are sometimes missing** (nothing between the semicolons)
- `rentals` — number of bikes rented in that hour: this is what we want to predict

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

import sys; sys.path.insert(0, "..")
from dnd_utils import TesterA

tester = TesterA()
content = tester.get_dataset_content()
print("Number of characters:", len(content))
print(content[:300])

## **A** · Loading the dataset

Convert `content` into the NumPy array `dataset_noisy`, shape `(n_samples, 8)`, dtype `np.float32`, columns in file order. `season` becomes the index of the season in the order `winter, spring, summer, autumn` (0–3). Missing values become `np.nan`.

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE

tester.test('dataset_load', dataset_noisy)

## **B** · Shuffling and splitting

Shuffle the rows with the permutation `np.random.default_rng(42).permutation(n_samples)` and split them 60 % / 20 % / 20 % into `dataset_split_train`, `dataset_split_val`, `dataset_split_test` (sizes of the first two: `int(n_samples * ratio)`).

Before you do: print the mean of `rentals` for the first and the last 400 rows of the unshuffled file. One comment line: why is shuffling necessary here?

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE

tester.test('dataset_split', dataset_split_train, dataset_split_val, dataset_split_test)

## **C** · Pre-processing

Create the **input** matrices `X_train`, `X_val`, `X_test` (`float32`, 13 columns) with exactly this layout:

| columns | content |
|---|---|
| 0–4 | `hour`, `temp_c`, `humidity`, `wind_kmh`, `weekday` — missing values replaced by the **median of that column in the training split**, then **standardised** with the mean and (population) standard deviation of the training split (computed after the replacement) |
| 5 | `holiday` (unchanged) |
| 6–9 | one-hot encoding of `season` (winter, spring, summer, autumn) |
| 10–12 | three indicator columns for `temp_c`, `humidity`, `wind_kmh`: `1` if the value was present, `0` if it was missing |

`rentals` must **not** be among the inputs. Nothing computed from the validation or test split may be used.

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE

tester.test('preprocess', X_train, X_val, X_test)

## **D** · Data iterators for the regression task

Label: `rentals` (column 7). Create the iterables `dataloader_reg_train`, `dataloader_reg_val`, `dataloader_reg_test`. Each step yields two `torch.float32` tensors of shapes `(batch_size, 13)` and `(batch_size, 1)`. Training and validation data are traversed in random order; the test iterator keeps the order of the test split.

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE

tester.test('reg_iter', dataloader_reg_train, dataloader_reg_val, dataloader_reg_test)

## **E** · The regression network

Define the network as a subclass of `torch.nn.Module` and assign an instance to `reg_model`: three fully connected layers with **64, 32 and 1** neurons, ReLU between the layers.

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE

tester.test('reg_model_architecture', reg_model)

## **F** · Training the regression network

- Train `reg_model` with the **MSE** loss; choose optimizer, learning rate and number of epochs yourself (judge by the validation curve: well trained, not severely over-fitted).
- Measure training and validation loss in every epoch and **plot both on one figure with a legend**; choose the y-range so the end of training is readable.
- Assign the average loss on the **full test set** to `test_mse`.
- Print predicted and true `rentals` for a few test samples.
- In one comment line: what is the MSE of always predicting the training mean, and how does your model compare?

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE

tester.test('reg_model_learning', test_mse)

## **G** · Data iterators for the classification task

Now we classify the **demand level** of each hour:

- category **0** ("low"): fewer than 25 rentals,
- category **1** ("medium"): at least 25 but fewer than 100,
- category **2** ("high"): 100 or more.

Inputs: the same 13 columns as before. Create `dataloader_cl_train`, `dataloader_cl_val`, `dataloader_cl_test` yielding tensors of shapes `(batch_size, 13)` (`torch.float32`) and `(batch_size,)` (`torch.int64`); random order for training and validation, original order for test. Print the class counts of the three splits.

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE

tester.test('cl_iter', dataloader_cl_train, dataloader_cl_val, dataloader_cl_test)

## **H** · The classification network

A `torch.nn.Module` subclass, instance in `cl_model`: three fully connected layers, the first two with **40 and 20** neurons and ReLU after each, the last one suitable for three-class classification.

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE

tester.test('cl_model_architecture', cl_model)

## **I** · Training the classifier with early stopping

- **Cross-entropy** loss; measure training and validation loss per epoch; plot both with a legend.
- **Early stopping**: stop when the validation loss has not improved for `patience` epochs, and **automatically restore the weights of the best epoch**.
- `test_ce`: average loss on the full test set; `test_acc`: accuracy on the test set.
- Print predicted and true categories for a few test samples.
- Compute the **confusion matrix** of the test set (`[i, j]` = number of samples of true category `i` predicted as `j`) and plot it: both axes labelled, category indices at the ticks, counts inside the cells.

In [ ]:
# implement your solution BELOW

# implement your solution ABOVE

tester.test('cl_model_learning', test_ce, test_acc)

---
## After the bell

Score yourself honestly:

| | done in time, tester OK | done, needed the docs | needed the solution |
|---|---|---|---|
| A load | | | |
| B split | | | |
| C preprocess | | | |
| D iterators | | | |
| E model | | | |
| F training + plot + test | | | |
| G labels + iterators | | | |
| H model | | | |
| I early stopping + confusion matrix | | | |

For every cell in the right-hand column, go back to the notebook that covers it (A–C → 04, D/G → 05, E/F → 06, H/I → 08 and 09), redo its blank-page drill tomorrow, and repeat this mock in a few days **from an empty copy**.